In [1]:
import numpy as np
import glob
import pandas as pd
import os


In [2]:
def create_sliding_windows(data, window_size = 400, step_size = 100):
    windows = []
    labels = []
    
    for i in range(0, len(data) - window_size + 1, step_size):
        window = data[i:i+window_size]  # Extract a window of specified size
        windows.append(window)

    return np.array(windows)

ROOT_DIR = r'C:\Codes\outdoor_terrain_classification_database\Measurements'

sensors = ['ACC','GYR']
terrain_labels ={
    'concrete':1 ,
    'grass': 2 ,
   'pebbles': 3,
    'sand': 4,
    'paving stone': 5,
    'synthetic running track': 6 ,
}

In [3]:
import random 

accelerometer_train_path_dict = dict()
gyroscope_train_path_dict = dict()

accelerometer_val_path_dict = dict()
gyroscope_val_path_dict = dict()

accelerometer_test_path_dict = dict()
gyroscope_test_path_dict = dict()

for item in terrain_labels.values():
    acc_type = glob.glob(ROOT_DIR + f'\\*ACC_{item}_2*')
    gyr_type = glob.glob(ROOT_DIR + f'\\*GYR_{item}_2*')

    acc_test = random.choice(acc_type)
    acc_type.pop(acc_type.index(acc_test))
    gyr_test = random.choice(gyr_type)
    gyr_type.pop(gyr_type.index(gyr_test))

    acc_val = random.sample(acc_type,2)
    gyr_val = random.sample(gyr_type,2)
    for i in range(2):
        acc_type.remove(acc_val[i])
        gyr_type.remove(gyr_val[i])


    accelerometer_train_path_dict[item] = acc_type
    gyroscope_train_path_dict[item] = gyr_type

    accelerometer_test_path_dict[item] = [acc_test]
    gyroscope_test_path_dict[item] = [gyr_test]

    accelerometer_val_path_dict[item] = acc_val
    gyroscope_val_path_dict[item] = gyr_val

In [4]:
print(f"Train length: {len(accelerometer_train_path_dict[1])}")
print(f"Test length: {len(accelerometer_test_path_dict[1])}")
print(f"Val length: {len(accelerometer_val_path_dict[1])}")


Train length: 4
Test length: 1
Val length: 2


In [5]:
# accelerometer_terrain_data_train = [[] for _ in range(3)]
# gyroscope_terrain_data_train = [[] for _ in range(3)]
# labels_list_train = []

# accelerometer_terrain_data_test = [[] for _ in range(3)]
# gyroscope_terrain_data_test = [[] for _ in range(3)]
# labels_list_test = []

# accelerometer_terrain_data_val = [[] for _ in range(3)]
# gyroscope_terrain_data_val = [[] for _ in range(3)]
# labels_list_val = []

def get_acc_gyr_data(accelerometer_path_dict,gyroscope_path_dict,step_size):
    
    accelerometer_terrain_data = [[] for _ in range(3)]
    gyroscope_terrain_data = [[] for _ in range(3)]
    labels_list = []

    for terrain_type in accelerometer_path_dict.keys():
        acc_paths = accelerometer_path_dict[terrain_type]
        gyr_paths = gyroscope_path_dict[terrain_type]

        for acc_path in acc_paths:
            df = pd.read_csv(acc_path)
            x = df.iloc[:,0]
            y = df.iloc[:,1]
            z = df.iloc[:,2]

            x_chunks  = create_sliding_windows(x,step_size=step_size)
            y_chunks = create_sliding_windows(y,step_size=step_size)
            z_chunks = create_sliding_windows(z,step_size=step_size)

            label_temp = [terrain_type for _ in range(len(z_chunks))]
            accelerometer_terrain_data[0] = [*accelerometer_terrain_data[0], *x_chunks]
            accelerometer_terrain_data[1] = [*accelerometer_terrain_data[1], *y_chunks]
            accelerometer_terrain_data[2] = [*accelerometer_terrain_data[2], *z_chunks]

            labels_list = [*labels_list, *label_temp]

        for gyr_path in gyr_paths:
            df = pd.read_csv(gyr_path)
            x = df.iloc[:,0]
            y = df.iloc[:,1]
            z = df.iloc[:,2]

            x_chunks  = create_sliding_windows(x,step_size=step_size)
            y_chunks = create_sliding_windows(y,step_size=step_size)
            z_chunks = create_sliding_windows(z,step_size=step_size)

            gyroscope_terrain_data[0] = [*gyroscope_terrain_data[0], *x_chunks]
            gyroscope_terrain_data[1] = [*gyroscope_terrain_data[1], *y_chunks]
            gyroscope_terrain_data[2] = [*gyroscope_terrain_data[2], *z_chunks]
    
    accelerometer_terrain_data = np.array(accelerometer_terrain_data)
    gyroscope_terrain_data = np.array(gyroscope_terrain_data)
    labels_list = np.array(labels_list)
            

    return accelerometer_terrain_data,gyroscope_terrain_data,labels_list

In [12]:
accelerometer_terrain_data_train,gyroscope_terrain_data_train,labels_list_train = get_acc_gyr_data(accelerometer_train_path_dict,gyroscope_train_path_dict,step_size=80)
accelerometer_terrain_data_test,gyroscope_terrain_data_test,labels_list_test = get_acc_gyr_data(accelerometer_test_path_dict,gyroscope_test_path_dict,step_size=120)
accelerometer_terrain_data_val,gyroscope_terrain_data_val,labels_list_val = get_acc_gyr_data(accelerometer_val_path_dict,gyroscope_val_path_dict,step_size=100)



In [14]:
from pathlib import Path

# Get the current working directory
current_directory = Path.cwd()

# Navigate to the parent directory
parent_directory = current_directory.parent

# Define the new directory inside the parent directory
new_directory = parent_directory / "numpy_time_series" / "three_d_proper"

# Create the directory if it doesn't exist
new_directory.mkdir(parents=True, exist_ok=True)

np.save(new_directory/'accelerometer_time_windows_xyz_train.npy', accelerometer_terrain_data_train) 
np.save(new_directory/'gyroscope_time_windows_xyz_train.npy', gyroscope_terrain_data_train) 
np.save(new_directory/'labels_train.npy', labels_list_train) 

np.save(new_directory/'accelerometer_time_windows_xyz_test.npy', accelerometer_terrain_data_test) 
np.save(new_directory/'gyroscope_time_windows_xyz_test.npy', gyroscope_terrain_data_test) 
np.save(new_directory/'labels_test.npy', labels_list_test) 

np.save(new_directory/'accelerometer_time_windows_xyz_val.npy', accelerometer_terrain_data_val) 
np.save(new_directory/'gyroscope_time_windows_xyz_val.npy', gyroscope_terrain_data_val) 
np.save(new_directory/'labels_val.npy', labels_list_val) 

In [13]:
print(f"Train len : {len(labels_list_train)}")
print(f"Val len: {len(accelerometer_terrain_data_val[0])}")
print(f"Test len: {len(accelerometer_terrain_data_test[0])}")

Train len : 432
Val len: 168
Test len: 72
